# 🖼️ Challenge 17: 이미지 처리 파이프라인 (Python + Go 하이브리드)

| 항목 | 내용 |
|------|------|
| **난이도** | ⭐⭐⭐⭐⭐ |
| **사전 지식** | 노트북 03 (RabbitMQ), 09 (동시성), Go 기초 |
| **소요 시간** | 60-90분 |
| **핵심 패턴** | Polyglot Microservice, RabbitMQ RPC, Redis 결과 저장 |

> ⚠️ **이 노트북은 Go 이미지 처리 서비스가 실행 중이어야 합니다!**  
> `docker compose up -d image-processor` 로 먼저 시작하세요.

---

📍 **네비게이션**: [← 16. 실시간 배달 알림](./16_challenge_realtime_delivery.ipynb) | **마지막 과제 🎉**

## 🎯 시나리오: 이미지 처리 마이크로서비스

### 왜 Go를 쓰나요?

| 작업 유형 | Python | Go |
|-----------|--------|----|
| **I/O 바운드** (API, DB) | ✅ asyncio로 충분 | ✅ goroutine |
| **CPU 바운드** (이미지, 암호화) | ❌ GIL 병목 | ✅ 네이티브 병렬 |
| **메모리 효율** | ❌ 높은 오버헤드 | ✅ 낮은 메모리 |
| **배포 크기** | ~150MB (Python + deps) | ~15MB (static binary) |

### 실제 사례
- **쿠팡**: 상품 이미지 리사이즈/썸네일 생성
- **인스타그램**: 이미지 필터/효과 적용
- **네이버**: 프로필 사진 리사이즈

> 💡 **Polyglot Architecture**: 각 언어의 강점을 활용!  
> Python = API/오케스트레이션, Go = CPU 집약 처리

## 🏗️ 아키텍처


<div align="center"> <svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 800 320" width="100%" height="auto"> <rect width="100%" height="100%" fill="#F8FAFC" rx="12" /> <text x="400" y="30" font-family="-apple-system, sans-serif" font-size="14" font-weight="bold" fill="#0F172A" text-anchor="middle">Python(FastAPI) &amp; Go 하이브리드 이미지 처리 파이프라인</text> <g transform="translate(40, 60)"> <rect x="0" y="0" width="200" height="210" rx="8" fill="#F1F5F9" stroke="#475569" stroke-width="1.5" /> <text x="100" y="24" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#1E293B" text-anchor="middle">Python FastAPI Server</text> <rect x="15" y="45" width="170" height="40" rx="4" fill="#FFFFFF" stroke="#CBD5E1" /> <text x="100" y="62" font-family="-apple-system, sans-serif" font-size="9" font-weight="bold" fill="#475569" text-anchor="middle">1. 이미지 URL 처리 요청</text> <text x="100" y="76" font-family="sans-serif" font-size="8" fill="#64748B" text-anchor="middle">URL: /images/process</text> <rect x="15" y="145" width="170" height="45" rx="4" fill="#E0F2FE" stroke="#0284C7" /> <text x="100" y="164" font-family="-apple-system, sans-serif" font-size="9" font-weight="bold" fill="#0369A1" text-anchor="middle">4. 처리 결과 확인</text> <text x="100" y="178" font-family="sans-serif" font-size="8" fill="#0C4A6E" text-anchor="middle">Redis 결과 조회 &amp; 완료 반환</text> </g> <path d="M 240 110 L 320 110" stroke="#0284C7" stroke-width="2" /><polygon points="314,106 320,110 314,114" fill="#0284C7" /> <text x="280" y="98" font-family="sans-serif" font-size="8" fill="#0369A1" text-anchor="middle">요청 전송</text> <path d="M 320 220 L 240 220" stroke="#15803D" stroke-width="2" /><polygon points="246,216 240,220 246,224" fill="#15803D" /> <text x="280" y="240" font-family="sans-serif" font-size="8" fill="#15803D" text-anchor="middle">완료 이벤트</text> <g transform="translate(330, 80)"> <rect x="0" y="0" width="140" height="60" rx="6" fill="#F3E8FF" stroke="#7E22CE" stroke-width="1.5" /> <text x="70" y="24" font-family="-apple-system, sans-serif" font-size="10" font-weight="bold" fill="#6B21A8" text-anchor="middle">RabbitMQ Queue</text> <text x="70" y="44" font-family="sans-serif" font-size="8" fill="#581C87" text-anchor="middle">image-processing-queue</text> </g> <g transform="translate(330, 180)"> <rect x="0" y="0" width="140" height="60" rx="6" fill="#F3E8FF" stroke="#7E22CE" stroke-width="1.5" /> <text x="70" y="24" font-family="-apple-system, sans-serif" font-size="10" font-weight="bold" fill="#6B21A8" text-anchor="middle">RabbitMQ Queue</text> <text x="70" y="44" font-family="sans-serif" font-size="8" fill="#581C87" text-anchor="middle">image-result-queue</text> </g> <path d="M 470 110 L 550 110" stroke="#0284C7" stroke-width="2" /><polygon points="544,106 550,110 544,114" fill="#0284C7" /> <path d="M 550 210 L 470 210" stroke="#15803D" stroke-width="2" /><polygon points="476,206 470,210 476,214" fill="#15803D" /> <g transform="translate(560, 60)"> <rect x="0" y="0" width="200" height="210" rx="8" fill="#ECFDF5" stroke="#10B981" stroke-width="2" /> <text x="100" y="24" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#047857" text-anchor="middle">Go image-processor (Worker)</text> <rect x="15" y="40" width="170" height="80" rx="4" fill="#FFFFFF" stroke="#A7F3D0" /> <text x="100" y="58" font-family="-apple-system, sans-serif" font-size="9" fill="#065F46" text-anchor="middle">2. 이미지 처리 가속 실행</text> <text x="100" y="74" font-family="sans-serif" font-size="8" fill="#047857" text-anchor="middle">- 다운로드 &amp; 리사이즈</text> <text x="100" y="90" font-family="sans-serif" font-size="8" fill="#047857" text-anchor="middle">- 필터 적용 &amp; 인코딩</text> <text x="100" y="106" font-family="sans-serif" font-size="8" fill="#047857" text-anchor="middle">- 초고속 동시 Go 루틴 처리</text> <rect x="15" y="140" width="170" height="50" rx="4" fill="#FFFFFF" stroke="#059669" stroke-width="1" /> <text x="100" y="158" font-family="-apple-system, sans-serif" font-size="9" font-weight="bold" fill="#047857" text-anchor="middle">3. 결과 캐싱 (Redis)</text> <text x="100" y="174" font-family="monospace" font-size="8" fill="#065F46" text-anchor="middle">image:result:{task_id}</text> </g> </svg> </div>


## 🛠️ Go 서비스 구조

```text
image-processor/
├── Dockerfile              # Multi-stage: golang:1.22-alpine → alpine (~15MB)
├── go.mod                  # 의존성: amqp091-go, go-redis, prometheus
├── cmd/
│   └── main.go              # 엔트리포인트: RabbitMQ Consumer + HTTP
└── internal/
    ├── handler/
    │   └── image.go         # 리사이즈/필터 로직 (Go 표준 라이브러리)
    └── config/
        └── config.go        # 환경변수 설정
```

### Go 서비스가 하는 일
1. RabbitMQ `image-processing-queue`에서 메시지 수신
2. 이미지 URL 다운로드 (또는 테스트 이미지 생성)
3. Nearest Neighbor 리사이즈 (Go 표준 `image` 패키지)
4. 필터 적용: grayscale, sepia, contrast, blur, sharpen
5. 결과를 Redis에 저장 (`image:result:{task_id}`)
6. 완료 이벤트를 RabbitMQ `image-result-queue`에 발행
7. `/health`, `/metrics` HTTP 엔드포인트 제공

## 📚 사용할 패턴 & API

| 패턴 | 브로커 | 용도 |
|------|--------|------|
| Direct Queue | RabbitMQ | 처리 요청/결과 전달 |
| Cache (K/V) | Redis | 처리 결과 저장/조회 |
| Health Check | HTTP | Go 서비스 상태 확인 |
| Prometheus | HTTP | 메트릭 수집 |

### API 엔드포인트
```text
FastAPI (:8000)
  POST /rabbitmq/direct/publish    → 이미지 처리 요청 발행
  GET  /redis/cache/get/{key}      → 처리 결과 조회

Go image-processor (:8081)
  GET  /health                     → 서비스 상태 확인
  GET  /metrics                    → Prometheus 메트릭
```

In [ ]:
# ===========================================
# 환경 설정 + Mock 데이터 로드
# ===========================================
import httpx
import asyncio
import json
import time
from pathlib import Path

# === 기본 설정 ===
BASE_URL = "http://localhost:8000"       # FastAPI
GO_SERVICE_URL = "http://localhost:8081"  # Go image-processor

# === Mock 데이터 로드 ===
mock_path = Path("../data/mock/images_metadata.json")
with open(mock_path, encoding="utf-8") as f:
    mock_data = json.load(f)

requests_data = mock_data["processing_requests"]
print(f"🖼️ 이미지 처리 요청: {len(requests_data)}건")
print(f"\n📝 샘플 요청:")
sample = requests_data[0]
for k, v in sample.items():
    print(f"  {k}: {v}")

## Step 1: Go 서비스 상태 확인 💚

### 왜 먼저 확인하나요?
Go 서비스가 실행 중이지 않으면 RabbitMQ에 메시지가 쌓이기만 하고 처리되지 않습니다.

```bash
# Go 서비스 실행 확인
docker compose ps image-processor

# 실행되지 않았다면:
docker compose up -d image-processor
```

In [ ]:
# ===========================================
# Step 1: Go 서비스 헬스체크
# ===========================================

async def check_go_service():
    """Go 이미지 처리 서비스의 상태를 확인합니다."""
    async with httpx.AsyncClient(timeout=5.0) as client:
        try:
            resp = await client.get(f"{GO_SERVICE_URL}/health")
            health = resp.json()
            print("✅ Go 이미지 처리 서비스 상태:")
            for k, v in health.items():
                status_icon = "✅" if v in ("healthy", "connected") else "❌"
                print(f"  {status_icon} {k}: {v}")
            return True
        except httpx.ConnectError:
            print("❌ Go 서비스에 연결할 수 없습니다!")
            print("\n💡 해결 방법:")
            print("  docker compose up -d image-processor")
            print("  # 10초 대기 후 다시 실행")
            return False

go_ready = await check_go_service()

## Step 2: Python에서 이미지 처리 요청 발행 📤

### 요청 메시지 구조 (JSON)
```json
{
  "task_id": "img_task_001",
  "image_url": "https://picsum.photos/id/1/1920/1080",
  "target_width": 800,
  "target_height": 600,
  "format": "jpeg",
  "quality": 85,
  "filters": ["grayscale"]
}
```

### 흐름
1. Python이 요청을 JSON으로 직렬화
2. RabbitMQ `image-processing-queue`에 발행
3. Go Consumer가 자동으로 수신 → 처리 시작

In [ ]:
# ===========================================
# Step 2: 이미지 처리 요청 발행 (첫 5건 테스트)
# ===========================================

async def publish_image_request(client, request_data):
    """
    이미지 처리 요청을 RabbitMQ에 발행합니다.
    FastAPI의 /rabbitmq/direct/publish API를 사용합니다.
    """
    resp = await client.post(
        f"{BASE_URL}/rabbitmq/direct/publish",
        params={"queue_name": "image-processing-queue"},
        json={
            "content": json.dumps(request_data),
            "metadata": {"durable": True}
        }
    )
    return resp.json()

# 첫 5건만 먼저 테스트
test_requests = requests_data[:5]

async with httpx.AsyncClient(timeout=10.0) as client:
    print("📤 이미지 처리 요청 발행 중... (5건 테스트)\n")
    
    for req in test_requests:
        result = await publish_image_request(client, req)
        filters_str = ", ".join(req.get("filters", [])) or "없음"
        print(f"  ✅ {req['task_id']}: {req['target_width']}x{req['target_height']} "
              f"({req['format']}) 필터=[{filters_str}]")

print(f"\n📨 5건 발행 완료! Go 서비스가 처리 중...")

## Step 3: Go 서비스 처리 결과 확인 🔍

### Go 서비스는 어떻게 처리하나요?

```text
메시지 수신 → JSON 파싱 → 이미지 다운로드 (또는 테스트 이미지 생성)
  → Nearest Neighbor 리사이즈 → 필터 적용 → JPEG/PNG 인코딩
  → Redis에 결과 저장 → RabbitMQ에 완료 이벤트 발행 → ACK
```

- 결과는 Redis에 2가지로 저장:
  - `image:result:{task_id}` → 메타데이터 (JSON)
  - `image:data:{task_id}` → 이미지 바이너리

In [ ]:
# ===========================================
# Step 3: Redis에서 처리 결과 조회
# ===========================================

async def get_result(client, task_id):
    """
    Redis에서 이미지 처리 결과를 조회합니다.
    Go 서비스가 image:result:{task_id} 키에 저장합니다.
    """
    key = f"image:result:{task_id}"
    resp = await client.get(f"{BASE_URL}/redis/cache/get/{key}")
    return resp.json()

# 5초 대기 후 결과 확인 (Go 서비스 처리 시간)
print("⏳ Go 서비스 처리 대기 중... (5초)")
await asyncio.sleep(5)

async with httpx.AsyncClient(timeout=10.0) as client:
    print("\n📊 처리 결과:\n")
    print(f"{'Task ID':<16} {'Status':<12} {'크기 변환':<22} {'시간(ms)':<10} {'용량'}")
    print("-" * 75)
    
    completed = 0
    for req in test_requests:
        result = await get_result(client, req["task_id"])
        data = result.get("data", {})
        
        if data and data.get("value"):
            try:
                info = json.loads(data["value"])
                size_str = f"{info.get('original_width', '?')}x{info.get('original_height', '?')} → {info.get('result_width', '?')}x{info.get('result_height', '?')}"
                bytes_str = f"{info.get('file_size_bytes', 0):,} bytes"
                ms_str = f"{info.get('processing_ms', 0):.1f}ms"
                status = info.get('status', 'unknown')
                icon = "✅" if status == "completed" else "❌"
                print(f"{icon} {req['task_id']:<14} {status:<12} {size_str:<22} {ms_str:<10} {bytes_str}")
                if status == "completed":
                    completed += 1
            except json.JSONDecodeError:
                print(f"⚠️  {req['task_id']:<14} 파싱 오류")
        else:
            print(f"⏳ {req['task_id']:<14} 아직 처리 중... (조금 더 기다리세요)")
    
    print(f"\n🎯 결과: {completed}/5건 처리 완료")

## Step 4: 전체 50건 동시 처리 테스트 🚀

### 왜 대량 테스트를 하나요?
- Go의 진짜 장점은 **동시 처리 성능**에서 드러남
- QoS 설정(prefetch=5)으로 동시 5건씩 병렬 처리
- Python GIL vs Go goroutine 성능 차이 체감

In [ ]:
# ===========================================
# Step 4: 50건 동시 발행 + 처리량 측정
# ===========================================

async def bulk_publish_and_measure(requests_list):
    """
    모든 요청을 동시에 발행하고 처리 시간을 측정합니다.
    """
    async with httpx.AsyncClient(timeout=30.0) as client:
        # === Phase 1: 전체 동시 발행 ===
        print(f"📤 {len(requests_list)}건 동시 발행 중...")
        publish_start = time.time()
        
        tasks = [publish_image_request(client, req) for req in requests_list]
        results = await asyncio.gather(*tasks, return_exceptions=True)
        
        publish_time = time.time() - publish_start
        success_count = sum(1 for r in results if not isinstance(r, Exception))
        print(f"✅ 발행 완료: {success_count}/{len(requests_list)}건 ({publish_time:.2f}초)")
        
        # === Phase 2: 결과 대기 ===
        print(f"\n⏳ Go 서비스 처리 대기 중...")
        
        # 폴링으로 결과 확인 (5초 간격, 최대 60초)
        process_start = time.time()
        completed_results = []
        
        for attempt in range(12):  # 최대 60초
            await asyncio.sleep(5)
            completed = 0
            
            for req in requests_list:
                key = f"image:result:{req['task_id']}"
                resp = await client.get(f"{BASE_URL}/redis/cache/get/{key}")
                data = resp.json().get("data", {})
                if data and data.get("value"):
                    try:
                        info = json.loads(data["value"])
                        if info.get("status") == "completed":
                            completed += 1
                    except json.JSONDecodeError:
                        pass
            
            elapsed = time.time() - process_start
            print(f"  [{elapsed:.0f}초] {completed}/{len(requests_list)}건 완료")
            
            if completed >= len(requests_list):
                break
        
        total_time = time.time() - process_start
        
        # === Phase 3: 결과 수집 ===
        all_results = []
        for req in requests_list:
            key = f"image:result:{req['task_id']}"
            resp = await client.get(f"{BASE_URL}/redis/cache/get/{key}")
            data = resp.json().get("data", {})
            if data and data.get("value"):
                try:
                    all_results.append(json.loads(data["value"]))
                except json.JSONDecodeError:
                    pass
        
        return all_results, publish_time, total_time

# 50건 전체 처리
all_results, pub_time, total_time = await bulk_publish_and_measure(requests_data)

## Step 5: 성능 분석 📊

### 측정 항목
- **처리량** (Throughput): 초당 처리 이미지 수
- **개별 처리 시간**: Go의 이미지당 ms
- **전체 파이프라인**: 발행 → 결과 조회까지 전체 시간

In [ ]:
# ===========================================
# Step 5: 성능 분석 결과
# ===========================================

if all_results:
    # 처리 시간 통계
    processing_times = [r.get("processing_ms", 0) for r in all_results if r.get("status") == "completed"]
    file_sizes = [r.get("file_size_bytes", 0) for r in all_results if r.get("status") == "completed"]
    completed_count = len(processing_times)
    
    print("=" * 60)
    print("📊 이미지 처리 파이프라인 성능 분석")
    print("=" * 60)
    
    print(f"\n🖼️ 처리 결과")
    print(f"  전체 요청: {len(requests_data)}건")
    print(f"  성공: {completed_count}건")
    print(f"  실패: {len(requests_data) - completed_count}건")
    
    if processing_times:
        avg_ms = sum(processing_times) / len(processing_times)
        min_ms = min(processing_times)
        max_ms = max(processing_times)
        
        print(f"\n⏱️ Go 개별 처리 시간")
        print(f"  평균: {avg_ms:.1f}ms")
        print(f"  최소: {min_ms:.1f}ms")
        print(f"  최대: {max_ms:.1f}ms")
        
        print(f"\n🚀 전체 파이프라인")
        print(f"  발행 시간: {pub_time:.2f}초")
        print(f"  처리 시간: {total_time:.2f}초")
        throughput = completed_count / total_time if total_time > 0 else 0
        print(f"  처리량: {throughput:.1f} images/sec")
        
        if file_sizes:
            avg_size = sum(file_sizes) / len(file_sizes)
            total_size = sum(file_sizes)
            print(f"\n📁 결과 파일")
            print(f"  평균 크기: {avg_size/1024:.1f}KB")
            print(f"  전체 크기: {total_size/1024/1024:.2f}MB")
    
    # === Python Pillow 예상 비교 ===
    print(f"\n\n{'='*60}")
    print("🔥 Go vs Python (Pillow) 예상 성능 비교")
    print("=" * 60)
    print(f"\n  │ {'':>15} │ {'Go':>12} │ {'Python*':>12} │ {'배율':>8} │")
    print(f"  ├{'':->17}┼{'':->14}┼{'':->14}┼{'':->10}┤")
    
    if processing_times:
        python_est = avg_ms * 5  # Python은 대략 5배 느림 (예상)
        print(f"  │ {'개별 처리':>10} │ {avg_ms:>9.1f}ms │ {python_est:>9.1f}ms │ {python_est/avg_ms:>6.1f}x │")
        
        go_throughput = throughput
        python_throughput = go_throughput / 5
        print(f"  │ {'처리량':>10} │ {go_throughput:>8.1f}/s │ {python_throughput:>8.1f}/s │ {go_throughput/python_throughput:>6.1f}x │")
    
    print(f"  │ {'메모리':>10} │ {'~15MB':>12} │ {'~150MB':>12} │ {'10x':>8} │")
    print(f"  │ {'동시성':>10} │ {'goroutine':>12} │ {'GIL 제한':>12} │ {'':>8} │")
    print(f"\n  * Python 수치는 예상치 (실제는 Pillow 벤치마크 필요)")
else:
    print("⚠️ 처리 결과가 없습니다. Go 서비스 상태를 확인하세요.")

## Step 6: Prometheus 메트릭 확인 📈

Go 서비스는 Prometheus 메트릭을 `/metrics` 엔드포인트로 노출합니다.

### 제공하는 메트릭
| 메트릭 | 타입 | 설명 |
|--------|------|------|
| `image_processor_images_total` | Counter | 처리된 이미지 수 (status 레이블) |
| `image_processor_duration_seconds` | Histogram | 처리 소요 시간 |
| `image_processor_in_flight` | Gauge | 현재 처리 중인 이미지 수 |

In [ ]:
# ===========================================
# Step 6: Prometheus 메트릭 조회
# ===========================================

async def check_metrics():
    """Go 서비스의 Prometheus 메트릭을 조회합니다."""
    async with httpx.AsyncClient(timeout=5.0) as client:
        try:
            resp = await client.get(f"{GO_SERVICE_URL}/metrics")
            metrics_text = resp.text
            
            print("📈 Go 이미지 처리 서비스 메트릭\n")
            
            # 필요한 메트릭만 필터링
            target_metrics = [
                "image_processor_images_total",
                "image_processor_duration_seconds",
                "image_processor_in_flight"
            ]
            
            for line in metrics_text.split("\n"):
                for target in target_metrics:
                    if line.startswith(target) and not line.startswith("#"):
                        print(f"  {line}")
            
            # 요약
            print(f"\n📝 Prometheus UI에서 확인:")
            print(f"  http://localhost:9090")
            print(f"  쿼리: image_processor_images_total")
            
        except httpx.ConnectError:
            print("⚠️ Go 서비스에 연결할 수 없습니다.")

await check_metrics()

## ✅ 결과 검증

### 체크리스트
- [ ] Go 서비스 `/health` 응답 정상
- [ ] RabbitMQ로 요청 발행 성공
- [ ] Redis에 결과 저장 확인
- [ ] 50건 동시 처리 성공
- [ ] Prometheus 메트릭 수집 확인

In [ ]:
# ===========================================
# 결과 검증 요약
# ===========================================

print("\n" + "=" * 60)
print("🏆 Challenge 17 결과 요약")
print("=" * 60)

checks = {
    "Go 서비스 연결": go_ready,
    "요청 발행 성공": len(all_results) > 0 if all_results else False,
    "전체 처리 완료": len(all_results) == len(requests_data) if all_results else False,
}

for check, passed in checks.items():
    icon = "✅" if passed else "❌"
    print(f"  {icon} {check}")

print(f"\n📝 통과 조건: 전체 50건 처리 성공 + Go 서비스 정상 동작")

## 💡 핵심 포인트 정리

### 1. Polyglot Architecture
> 한 가지 언어로 모든 걸 할 필요 없다!  
> 각 언어의 강점을 활용: Python=API/오케스트레이션, Go=CPU 집약

### 2. 메시지 큐는 마이크로서비스의 접착제
> RabbitMQ가 Python과 Go 사이의 통신을 담당  
> 두 서비스는 서로를 직접 알 필요 없음 (loose coupling)

### 3. Redis = 공유 상태 저장소
> Go가 처리하고 Redis에 저장, Python이 Redis에서 조회  
> 언어/서비스 독립적으로 데이터 공유

### 4. Go의 이미지 처리 장점
- GIL 없음 → 진정한 병렬 처리
- 낮은 메모리 사용량 (~15MB vs ~150MB)
- 정적 바이너리 → 빠른 배포, 작은 컨테이너

### 5. 관측 가능성 (Observability)
> Health check + Prometheus metrics + 구조화된 로그  
> 마이크로서비스는 반드시 관측 가능해야 한다!

## 🚀 확장 아이디어

### 기능 확장
- **WebP 지원**: Go에 `golang.org/x/image/webp` 추가
- **CDN 연동**: 처리된 이미지를 S3 → CloudFront로 배포
- **썸네일 생성**: 하나의 원본에서 여러 크기 자동 생성
- **워터마크**: 이미지에 로고/텍스트 오버레이

### Production 고려사항
- **수평 확장**: Go 서비스 레플리카 증가 (`docker compose scale image-processor=3`)
- **Circuit Breaker**: 다운로드 실패 시 회로 차단
- **Rate Limiting**: 동시 처리 수 제한 (QoS prefetch)
- **DLQ**: 처리 실패 메시지 격리 + 관리자 알림
- **이미지 크기 제한**: 악의적 대용량 이미지 차단

In [ ]:
# ===========================================
# 리소스 정리
# ===========================================

# 테스트용 Redis 데이터 정리
async def cleanup():
    async with httpx.AsyncClient(timeout=10.0) as client:
        cleaned = 0
        for req in requests_data:
            task_id = req["task_id"]
            # 결과 메타데이터 삭제
            await client.delete(f"{BASE_URL}/redis/cache/delete/image:result:{task_id}")
            # 이미지 데이터 삭제
            await client.delete(f"{BASE_URL}/redis/cache/delete/image:data:{task_id}")
            cleaned += 1
        print(f"🧹 Redis 정리 완료: {cleaned}건의 결과 데이터 삭제")

# 정리 실행 (필요 시 주석 해제)
# await cleanup()
print("💡 정리하려면 위 주석을 해제하세요")

---

## 🎉 모든 과제 완료!

17개의 노트북을 모두 완료했습니다. 학습 로드맵을 돌아보면:

| Phase | 노트북 | 학습 내용 |
|-------|---------|----------|
| **1. 기본** | 01-04 | Redis, RabbitMQ, Kafka 개별 심화 |
| **2. 분석** | 05-06 | 벤치마크 + 모니터링 |
| **3. 고급** | 07-10 | DLQ, 신뢰성, 동시성, Saga |
| **4. 실전** | 11-18 | 결제, 예매, 채팅, 대용량, Saga, 배달, Go 하이브리드, **AI 시맨틱 캐시** |

### 다음 단계
- [→ 18. AI 시맨틱 캐시 파이프라인](./18_challenge_ai_semantic_cache.ipynb) — Redis Vector Set + Kafka + RabbitMQ
- 이 프로젝트를 포트폴리오에 활용하세요
- 실제 클라우드 환경(AWS/GCP)에 배포해보세요
- Kubernetes로 오케스트레이션을 확장해보세요

---

📍 **네비게이션**: [← 16. 실시간 배달 알림](./16_challenge_realtime_delivery.ipynb) | [→ 18. AI 시맨틱 캐시](./18_challenge_ai_semantic_cache.ipynb) | [🏠 프로젝트 개요](./01_project_overview.ipynb)